# Experiment 9 metric figures

Generate publication-ready figures from the committed Experiment 9 evidence. This notebook reads predictions, metrics, and training history only; it does not load dataset images or model weights.

Outputs: confusion matrix, ROC/precision-recall panel, case-level metrics, training curves, calibration plot, and a case-level metrics CSV. Results describe a reused internal eight-case holdout and are not clinical validation.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'blue': '#176B87', 'orange': '#F28E2B', 'green': '#2A9D8F',
          'red': '#D1495B', 'purple': '#7B61A8', 'gray': '#6C757D'}
DPI = 300

## Locate and validate committed evidence

In [ ]:
def find_experiment_root() -> Path:
    """Return the Experiment 9 directory from common launch locations."""
    candidates = [Path.cwd(), Path.cwd() / 'experiments' / 'exp-9']
    for candidate in candidates:
        if (candidate / 'outputs' / 'resnet50_test_predictions.csv').is_file():
            return candidate
    raise FileNotFoundError('Run from the repository root or experiments/exp-9 directory.')

ROOT = find_experiment_root()
EVIDENCE_DIR = ROOT / 'outputs'
FIGURE_DIR = ROOT / 'metric_figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

predictions = pd.read_csv(EVIDENCE_DIR / 'resnet50_test_predictions.csv')
metrics = json.loads((EVIDENCE_DIR / 'resnet50_test_metrics.json').read_text(encoding='utf-8'))
history = json.loads((EVIDENCE_DIR / 'resnet50_training_history.json').read_text(encoding='utf-8'))

required = {'relative_path', 'case_id', 'slide_id', 'label', 'probability', 'prediction'}
assert required.issubset(predictions.columns)
assert len(predictions) == metrics['patches'] == 5447
assert predictions['case_id'].nunique() == metrics['cases'] == 8
assert not predictions['relative_path'].duplicated().any()
assert predictions['label'].isin([0, 1]).all()
assert predictions['probability'].between(0, 1).all()
threshold = float(metrics['threshold'])
expected_prediction = (predictions['probability'].to_numpy() >= threshold).astype(int)
assert np.array_equal(expected_prediction, predictions['prediction'].to_numpy(dtype=int))
print(f'Evidence: {len(predictions):,} patches, {predictions.case_id.nunique()} cases')
print('Figure directory:', FIGURE_DIR.resolve())

## Recompute pooled and case-level metrics

In [ ]:
def safe_ratio(numerator: int, denominator: int) -> float:
    """Return a ratio or NaN when its denominator is zero."""
    return numerator / denominator if denominator else np.nan

def calculate_case_metrics(frame: pd.DataFrame) -> dict:
    """Calculate one case's metrics without inventing single-class values."""
    y_true = frame['label'].to_numpy(dtype=int)
    y_pred = frame['prediction'].to_numpy(dtype=int)
    probability = frame['probability'].to_numpy(dtype=float)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    sensitivity = safe_ratio(tp, tp + fn)
    specificity = safe_ratio(tn, tn + fp)
    return {
        'patches': len(frame),
        'tumour_prevalence': float(y_true.mean()),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': (sensitivity + specificity) / 2 if has_both_classes else np.nan,
        'precision': precision_score(y_true, y_pred, zero_division=0) if y_true.sum() else np.nan,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'f1': f1_score(y_true, y_pred, zero_division=0) if y_true.sum() else np.nan,
        'roc_auc': roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }

case_metrics = pd.DataFrame([
    {'case_id': case_id, **calculate_case_metrics(group)}
    for case_id, group in predictions.groupby('case_id', sort=True)
]).set_index('case_id')
case_metrics.to_csv(FIGURE_DIR / 'case_level_metrics.csv')

y_true = predictions['label'].to_numpy(dtype=int)
y_pred = predictions['prediction'].to_numpy(dtype=int)
probability = predictions['probability'].to_numpy(dtype=float)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
recomputed = {
    'accuracy': accuracy_score(y_true, y_pred),
    'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
    'precision': precision_score(y_true, y_pred),
    'recall': recall_score(y_true, y_pred),
    'f1': f1_score(y_true, y_pred),
    'roc_auc': roc_auc_score(y_true, probability),
}
assert all(np.isclose(recomputed[name], metrics[name]) for name in recomputed)
assert cm.tolist() == metrics['confusion_matrix']
display(case_metrics.round(4))

## 1. Confusion matrix

In [ ]:
row_percent = cm / cm.sum(axis=1, keepdims=True) * 100
fig, ax = plt.subplots(figsize=(6.2, 5.2))
image = ax.imshow(row_percent, cmap='Blues', vmin=0, vmax=100)
for row in range(2):
    for column in range(2):
        color = 'white' if row_percent[row, column] > 55 else '#17202A'
        ax.text(column, row, f'{cm[row, column]:,}\n{row_percent[row, column]:.1f}%',
                ha='center', va='center', fontsize=13, color=color, fontweight='bold')
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['Non-tumour', 'Tumour'],
       yticklabels=['Non-tumour', 'Tumour'], xlabel='Predicted class', ylabel='Actual class',
       title=f'Experiment 9 confusion matrix (threshold = {threshold:.1f})')
ax.grid(False)
fig.colorbar(image, ax=ax, label='Percentage within actual class (%)')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'confusion_matrix.png', dpi=DPI, bbox_inches='tight')
plt.show()

## 2. ROC and precision-recall curves

In [ ]:
fpr, tpr, _ = roc_curve(y_true, probability)
precision_curve, recall_curve, _ = precision_recall_curve(y_true, probability)
roc_auc = roc_auc_score(y_true, probability)
average_precision = average_precision_score(y_true, probability)
tn, fp, fn, tp = cm.ravel()
operating_fpr = fp / (fp + tn)
operating_tpr = tp / (tp + fn)
operating_precision = tp / (tp + fp)
prevalence = y_true.mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, color=COLORS['blue'], linewidth=2.2, label=f'ROC-AUC = {roc_auc:.4f}')
axes[0].plot([0, 1], [0, 1], '--', color=COLORS['gray'], label='Chance')
axes[0].scatter(operating_fpr, operating_tpr, color=COLORS['red'], s=65, zorder=3,
                label=f'Threshold {threshold:.1f}')
axes[0].set(xlabel='False-positive rate', ylabel='True-positive rate', title='ROC curve',
            xlim=(0, 1), ylim=(0, 1.01))
axes[0].legend(loc='lower right')

axes[1].plot(recall_curve, precision_curve, color=COLORS['orange'], linewidth=2.2,
             label=f'Average precision = {average_precision:.4f}')
axes[1].axhline(prevalence, linestyle='--', color=COLORS['gray'],
                label=f'Prevalence = {prevalence:.3f}')
axes[1].scatter(operating_tpr, operating_precision, color=COLORS['red'], s=65, zorder=3,
                label=f'Threshold {threshold:.1f}')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-recall curve',
            xlim=(0, 1), ylim=(0, 1.01))
axes[1].legend(loc='lower left')
fig.suptitle('Experiment 9 patch-level discrimination on the reused holdout', fontsize=14)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'roc_precision_recall_curves.png', dpi=DPI, bbox_inches='tight')
plt.show()

## 3. Case-level performance

In [ ]:
plot_metrics = {
    'accuracy': ('Accuracy', COLORS['blue']),
    'balanced_accuracy': ('Balanced accuracy', COLORS['orange']),
    'sensitivity': ('Sensitivity', COLORS['green']),
    'specificity': ('Specificity', COLORS['red']),
    'f1': ('F1', COLORS['purple']),
}
x = np.arange(len(case_metrics))
offsets = np.linspace(-0.24, 0.24, len(plot_metrics))
fig, ax = plt.subplots(figsize=(13, 6))
for offset, (column, (label, color)) in zip(offsets, plot_metrics.items()):
    values = case_metrics[column].to_numpy(dtype=float)
    valid = np.isfinite(values)
    ax.scatter(x[valid] + offset, values[valid], s=55, color=color, label=label, zorder=3)
for index, patches in enumerate(case_metrics['patches']):
    ax.text(index, 1.015, f'n={patches:,}', ha='center', va='bottom', fontsize=8, rotation=45)
ax.set_xticks(x, [case.replace('TCGA-', '') for case in case_metrics.index], rotation=40, ha='right')
ax.set(ylim=(0, 1.08), ylabel='Metric value', xlabel='Held-out TCGA case',
       title='Case-level performance (missing dots are undefined for single-class cases)')
ax.legend(ncol=5, loc='lower center', bbox_to_anchor=(0.5, -0.38))
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'case_level_metrics.png', dpi=DPI, bbox_inches='tight')
plt.show()

## 4. Fixed-schedule training curves

In [ ]:
head_epochs = len(history['head']['loss'])
fine_epochs = len(history['fine_tuning']['loss'])
epochs = np.arange(1, head_epochs + fine_epochs + 1)
accuracy = history['head']['accuracy'] + history['fine_tuning']['accuracy']
loss = history['head']['loss'] + history['fine_tuning']['loss']

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
axes[0].plot(epochs, accuracy, marker='o', color=COLORS['blue'], linewidth=2)
axes[0].set(xlabel='Epoch', ylabel='Training accuracy', title='Training accuracy', xticks=epochs)
axes[1].plot(epochs, loss, marker='o', color=COLORS['orange'], linewidth=2)
axes[1].set(xlabel='Epoch', ylabel='Training loss', title='Training loss', xticks=epochs)
for ax in axes:
    ax.axvline(head_epochs + 0.5, color=COLORS['red'], linestyle='--', linewidth=1.5)
    ax.text(head_epochs + 0.55, ax.get_ylim()[1], 'Fine-tuning begins', color=COLORS['red'],
            va='top', fontsize=9)
fig.suptitle('Experiment 9 fixed 2+3 epoch schedule (training diagnostics only)', fontsize=14)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'training_curves.png', dpi=DPI, bbox_inches='tight')
plt.show()

## 5. Probability calibration

In [ ]:
observed, predicted = calibration_curve(y_true, probability, n_bins=10, strategy='quantile')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), gridspec_kw={'width_ratios': [1.25, 1]})
axes[0].plot([0, 1], [0, 1], '--', color=COLORS['gray'], label='Perfect calibration')
axes[0].plot(predicted, observed, marker='o', color=COLORS['green'], linewidth=2,
             label='ResNet50')
axes[0].set(xlabel='Mean predicted probability', ylabel='Observed tumour fraction',
            title='Quantile calibration curve', xlim=(0, 1), ylim=(0, 1))
axes[0].legend(loc='upper left')
axes[1].hist(probability, bins=np.linspace(0, 1, 21), color=COLORS['blue'], alpha=0.85)
axes[1].axvline(threshold, color=COLORS['red'], linestyle='--', label=f'Threshold {threshold:.1f}')
axes[1].set(xlabel='Predicted tumour probability', ylabel='Patch count',
            title='Prediction distribution')
axes[1].legend()
fig.suptitle('Experiment 9 patch-level probability calibration', fontsize=14)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'calibration_plot.png', dpi=DPI, bbox_inches='tight')
plt.show()

## 6. Patient-by-metric heatmap

In [ ]:
heatmap_columns = {
    'accuracy': 'Accuracy',
    'balanced_accuracy': 'Balanced accuracy',
    'precision': 'Precision',
    'sensitivity': 'Sensitivity',
    'specificity': 'Specificity',
    'f1': 'F1',
    'roc_auc': 'ROC-AUC',
}
heatmap_frame = case_metrics[list(heatmap_columns)].copy()
case_macro_mean = heatmap_frame.mean(axis=0, skipna=True)
heatmap_values = np.vstack([heatmap_frame.to_numpy(dtype=float),
                            case_macro_mean.to_numpy(dtype=float)])
masked_values = np.ma.masked_invalid(heatmap_values)
cmap = plt.colormaps['RdYlGn'].copy()
cmap.set_bad('#D9D9D9')
metadata_rows = []
for case_id, row in case_metrics.iterrows():
    single_class = row['tumour_prevalence'] in (0.0, 1.0)
    status = 'single ground-truth class' if single_class else 'both ground-truth classes'
    metadata_rows.append([case_id.replace('TCGA-', ''), f'{int(row["patches"]):,}', status])
metadata_rows.append(['CASE-MACRO MEAN', '8 cases', 'available cases'])

headers = ['Patient', 'Patches', 'Class composition', *heatmap_columns.values()]
cell_text = []
cell_colors = []
for row_index, metadata in enumerate(metadata_rows):
    metric_labels = []
    metric_colors = []
    for value in heatmap_values[row_index]:
        metric_labels.append('N/A' if not np.isfinite(value) else f'{value:.3f}')
        metric_colors.append('#D9D9D9' if not np.isfinite(value) else cmap((value - 0.5) / 0.5))
    metadata_color = '#DCE6F1' if row_index < len(case_metrics) else '#C7D5E5'
    cell_text.append([*metadata, *metric_labels])
    cell_colors.append([metadata_color] * 3 + metric_colors)

fig, ax = plt.subplots(figsize=(16, 6.8))
ax.axis('off')
table = ax.table(
    cellText=cell_text, cellColours=cell_colors, colLabels=headers,
    colColours=['#294C60'] * len(headers), cellLoc='center', colLoc='center',
    colWidths=[0.14, 0.075, 0.15, *([0.085] * len(heatmap_columns))],
    bbox=[0.0, 0.02, 0.94, 0.90],
)
table.auto_set_font_size(False)
table.set_fontsize(9.5)
mean_table_row = len(metadata_rows)
for (row, column), cell in table.get_celld().items():
    cell.set_edgecolor('white')
    cell.set_linewidth(1.5)
    if row == 0:
        cell.set_text_props(color='white', weight='bold')
        cell.set_height(0.085)
    else:
        cell.set_height(0.082)
        if column >= 3:
            value = heatmap_values[row - 1, column - 3]
            if np.isfinite(value) and (value < 0.62 or value > 0.94):
                cell.set_text_props(color='white', weight='bold')
            else:
                cell.set_text_props(color='#17202A', weight='bold')
        if row == mean_table_row:
            cell.set_linewidth(2.5)
            cell.set_edgecolor('#294C60')
            cell.set_text_props(weight='bold')
ax.set_title('Experiment 9 patient-level metric table', fontsize=16, pad=14)
normalizer = plt.Normalize(vmin=0.5, vmax=1.0)
mappable = plt.cm.ScalarMappable(norm=normalizer, cmap=cmap)
mappable.set_array([])
colorbar = fig.colorbar(mappable, ax=ax, fraction=0.025, pad=0.01)
colorbar.set_label('Metric value')
fig.text(0.5, 0.01, 'Case-macro mean gives each patient equal weight regardless of patch count; gray cells are undefined.',
         ha='center', fontsize=9, color=COLORS['gray'])
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(FIGURE_DIR / 'patient_metric_heatmap.png', dpi=DPI, bbox_inches='tight')
plt.show()

## 7. Case-level metric boxplots

In [ ]:
metric_groups = {
    'Discrimination': {'roc_auc': 'ROC-AUC'},
    'Detection': {'sensitivity': 'Sensitivity', 'specificity': 'Specificity'},
    'Classification': {
        'accuracy': 'Accuracy', 'f1': 'F1', 'precision': 'Precision',
        'balanced_accuracy': 'Balanced accuracy',
    },
}
group_colors = [COLORS['purple'], COLORS['green'], COLORS['blue']]
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True,
                         gridspec_kw={'height_ratios': [1.1, 2.0, 3.7], 'hspace': 0.28})
for ax, (group_name, columns), group_color in zip(axes, metric_groups.items(), group_colors):
    data = [case_metrics[column].dropna().to_numpy(dtype=float) for column in columns]
    positions = np.arange(1, len(columns) + 1)
    artists = ax.boxplot(
        data, vert=False, positions=positions, widths=0.52, patch_artist=True, showmeans=True,
        meanprops={'marker': 'D', 'markerfacecolor': COLORS['red'],
                   'markeredgecolor': 'white', 'markersize': 6},
        medianprops={'color': '#17202A', 'linewidth': 1.8},
    )
    for patch in artists['boxes']:
        patch.set_facecolor(group_color)
        patch.set_alpha(0.68)
    for position, values in zip(positions, data):
        jitter = np.linspace(-0.10, 0.10, len(values)) if len(values) > 1 else np.array([0.0])
        ax.scatter(values, position + jitter, color=COLORS['orange'], s=28,
                   edgecolor='white', linewidth=0.5, zorder=3)
        ax.text(0.505, position, f'n={len(values)}', ha='left', va='center',
                fontsize=8, color=COLORS['gray'])
    ax.set_yticks(positions, list(columns.values()))
    ax.invert_yaxis()
    ax.set_xlim(0.5, 1.01)
    ax.set_title(group_name, loc='left', fontsize=12, fontweight='bold', color=group_color)
    ax.grid(axis='x', alpha=0.35)
    ax.grid(axis='y', visible=False)
axes[-1].set_xlabel('Metric value across held-out cases')
mean_handle, = axes[0].plot([], [], marker='D', linestyle='None',
                            color=COLORS['red'], label='Case-macro mean')
case_handle, = axes[0].plot([], [], marker='o', linestyle='None',
                            color=COLORS['orange'], label='Patient case')
fig.legend(handles=[mean_handle, case_handle], loc='lower center', ncol=2,
           bbox_to_anchor=(0.5, 0.005))
fig.suptitle('Experiment 9 case-level metric distributions', fontsize=16)
fig.subplots_adjust(top=0.90, bottom=0.10, left=0.16, right=0.98, hspace=0.36)
fig.savefig(FIGURE_DIR / 'case_metric_boxplots.png', dpi=DPI, bbox_inches='tight')
plt.show()

## Output inventory

In [ ]:
expected_outputs = [
    'confusion_matrix.png',
    'roc_precision_recall_curves.png',
    'case_level_metrics.png',
    'training_curves.png',
    'calibration_plot.png',
    'patient_metric_heatmap.png',
    'case_metric_boxplots.png',
    'case_level_metrics.csv',
]
for name in expected_outputs:
    path = FIGURE_DIR / name
    assert path.is_file() and path.stat().st_size > 0, path
    print(f'{name}: {path.stat().st_size:,} bytes')